**Question:** What does the raw singles-match export look like, and what data-quality problems must later steps handle?

**Assumptions:** Official singles matches only (SM/SF) from 2017-05 onward; pseudonymized export.

**Method:** Descriptive EDA of match counts, ratings, ages, venues and head-to-head sparsity, with no modelling.

**Result:** 17,128 matches; birth-year bounds [1930, 2015], cleaned venues, and ~70% H2H sparsity are carried into feature engineering.

# 01 — Data Exploration (EDA)

Loads `../data/matches_export.csv` — 17,128 singles-league matches (`LigaTip IN ('SM','SF')`,
Masters/friendlies/doubles excluded, `Datum >= 2017-05-01`). See `../data/README.md` for the
exporter details and the empirical data-quality findings this notebook builds on.

**Scope: EDA only.** No feature engineering (that's `02_feature_engineering.ipynb`), no model
training (that's `03_model_training.ipynb`).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.grid'] = True

df = pd.read_csv('../data/matches_export.csv', parse_dates=['Datum'])
df['Datum'] = pd.to_datetime(df['Datum']).dt.tz_localize(None)
print(df.shape)
df.head(3)


## Shape, dtypes, date range

In [ ]:
print(f"Rows: {len(df):,}   Columns: {df.shape[1]}")
print(f"Date range: {df['Datum'].min().date()} .. {df['Datum'].max().date()}")
print()
df.dtypes


## Match volume over time

In [ ]:
per_season = df.groupby('SezonaId').agg(
    Datum_min=('Datum', 'min'),
    n_matches=('SusretId', 'count'),
).sort_values('Datum_min')

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(per_season['Datum_min'], per_season['n_matches'], marker='o', ms=3, lw=1)
ax.set_title('Matches per season (chronological)')
ax.set_ylabel('# matches')
ax.set_xlabel('Season start date')
plt.tight_layout()
plt.show()

per_month = df.set_index('Datum').resample('MS').size()
fig, ax = plt.subplots(figsize=(12, 4))
per_month.plot(ax=ax)
ax.set_title('Matches per calendar month')
ax.set_ylabel('# matches')
plt.tight_layout()
plt.show()

df['LigaTip'].value_counts()


Matches per gender (SM vs SF) per season:

In [ ]:
gender_season = df.groupby(['SezonaId', 'LigaTip']).size().unstack(fill_value=0)
gender_season = gender_season.join(per_season['Datum_min']).sort_values('Datum_min')

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(gender_season['Datum_min'], gender_season.get('SM', 0), label='SM (men)')
ax.plot(gender_season['Datum_min'], gender_season.get('SF', 0), label='SF (women)')
ax.set_title('Matches per season by gender league type')
ax.legend()
plt.tight_layout()
plt.show()

print(f"SM matches: {(df['LigaTip']=='SM').sum():,} ({(df['LigaTip']=='SM').mean():.1%})")
print(f"SF matches: {(df['LigaTip']=='SF').sum():,} ({(df['LigaTip']=='SF').mean():.1%})")


## ELO distribution

In [ ]:
elo_long = pd.concat([
    df[['WinnerEloBefore']].rename(columns={'WinnerEloBefore': 'EloBefore'}),
    df[['LoserEloBefore']].rename(columns={'LoserEloBefore': 'EloBefore'}),
])
print(elo_long['EloBefore'].describe())

fig, ax = plt.subplots()
ax.hist(df['WinnerEloBefore'], bins=60, alpha=0.5, label='Winner EloBefore')
ax.hist(df['LoserEloBefore'], bins=60, alpha=0.5, label='Loser EloBefore')
ax.set_title('EloBefore distribution (winner-side vs loser-side rows)')
ax.legend()
plt.tight_layout()
plt.show()


## Class balance / naive baselines

At this stage the CSV is one row per match (winner/loser columns) — there is no single
binary label to be "imbalanced" yet. The duplicate-and-flip step in `02_feature_engineering.ipynb`
is what produces the actual 50/50-by-construction training label. What IS meaningful here is
checking how often the pre-match Elo favorite actually wins — this previews Experiment 0
(the closed-form Elo baseline) before any model is trained.


In [ ]:
df['EloDiffWinnerMinusLoser'] = df['WinnerEloBefore'] - df['LoserEloBefore']
favorite_won = (df['EloDiffWinnerMinusLoser'] > 0).mean()
print(f"Share of matches where the WINNER already had the higher EloBefore: {favorite_won:.1%}")
print("(naive 'always back the Elo favorite' baseline accuracy)")

fig, ax = plt.subplots()
ax.hist(df['EloDiffWinnerMinusLoser'], bins=60)
ax.axvline(0, color='k', lw=1)
ax.set_title('EloBefore(winner) - EloBefore(loser)')
plt.tight_layout()
plt.show()

bins = [0, 25, 50, 100, 150, 200, 300, 1000]
tmp = df.copy()
tmp['abs_diff'] = tmp['EloDiffWinnerMinusLoser'].abs()
tmp['higher_rated_won'] = tmp['EloDiffWinnerMinusLoser'] > 0
tmp['bin'] = pd.cut(tmp['abs_diff'], bins)
print(tmp.groupby('bin', observed=True)['higher_rated_won'].agg(['mean', 'count']))


## H2H sparsity

In [ ]:
h2h_zero_pct = (df['H2HTotalBefore'] == 0).mean()
print(f"Share of matches with ZERO prior H2H meetings between the two players: {h2h_zero_pct:.1%}")
print(df['H2HTotalBefore'].describe())

fig, ax = plt.subplots()
df['H2HTotalBefore'].clip(upper=10).value_counts().sort_index().plot(kind='bar', ax=ax)
ax.set_title('Prior H2H meeting count before this match (clipped at 10+)')
plt.tight_layout()
plt.show()


## LigaKoeficjent / razina distribution

In [ ]:
print(df['LigaKoeficjent'].value_counts().sort_index())

fig, ax = plt.subplots()
df['LigaKoeficjent'].value_counts().sort_index().plot(kind='bar', ax=ax)
ax.set_title('Matches per LigaKoeficjent (razina)')
plt.tight_layout()
plt.show()

elo_by_koef = pd.concat([
    df[['LigaKoeficjent', 'WinnerEloBefore']].rename(columns={'WinnerEloBefore': 'Elo'}),
    df[['LigaKoeficjent', 'LoserEloBefore']].rename(columns={'LoserEloBefore': 'Elo'}),
]).groupby('LigaKoeficjent')['Elo'].mean()
print(elo_by_koef)
print("\nLigaKoeficjent=1 has the HIGHEST average Elo -> koef 1 is the TOP tier, higher koef = lower tier.")


## Lokacija (venue) distribution

In [ ]:
loc_counts = df['Lokacija'].value_counts()
print(f"Distinct Lokacija values: {df['Lokacija'].nunique()}")
print(loc_counts)

fig, ax = plt.subplots(figsize=(10, 5))
loc_counts.plot(kind='barh', ax=ax)
ax.invert_yaxis()
ax.set_title('Matches per Lokacija (venue)')
plt.tight_layout()
plt.show()

norm_nunique = df['Lokacija'].str.strip().str.lower().nunique()
print(f"Distinct values after lower/trim normalization: {norm_nunique} (same as raw -> no messy free-text variants)")


## Season 'pyramid size' — BrojLigaUSezoni / BrojIgracaUkupno per season and gender

In [ ]:
pop = df.groupby(['SezonaId', 'LigaTip'])[['BrojLigaUSezoni', 'BrojIgracaUkupno']].first().reset_index()
pop = pop.merge(per_season['Datum_min'], on='SezonaId').sort_values('Datum_min')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for tip, g in pop.groupby('LigaTip'):
    axes[0].plot(g['Datum_min'], g['BrojLigaUSezoni'], marker='o', ms=3, label=tip)
    axes[1].plot(g['Datum_min'], g['BrojIgracaUkupno'], marker='o', ms=3, label=tip)
axes[0].set_title('BrojLigaUSezoni per season, by gender')
axes[1].set_title('BrojIgracaUkupno per season, by gender')
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()

print(pop.groupby('LigaTip')[['BrojLigaUSezoni', 'BrojIgracaUkupno']].describe())


**Important caveat for notebook 02 (relative pyramid position feature):** `BrojLigaUSezoni`
counts the number of *distinct `LigaId` groups* active that season/gender (e.g. "3A / 3B / 3C"
are three separate parallel groups at the same `LigaKoeficjent`), **not** the number of skill
*levels* — `LigaKoeficjent` itself only ranges 1-8 (see the razina distribution above). SM alone
averages ~24 distinct `LigaId` groups per season vs. only 8 possible `LigaKoeficjent` levels. So
`LigaKoeficjent / BrojLigaUSezoni` is a rough depth-normalized proxy, not a clean top-to-bottom
percentile — notebook 02 computes both the originally planned literal ratio AND a cleaner empirical percentile
using the actual min/max `LigaKoeficjent` observed that season, and keeps both, clearly labeled.


## Missing-value / quality audit

In [ ]:
for col in ['WinnerDaysSinceLastMatch', 'LoserDaysSinceLastMatch']:
    n_null = df[col].isna().sum()
    print(f"{col}: {n_null} nulls ({n_null/len(df):.2%}) -- player's FIRST-EVER in-scope match, no LAG available")

for col in ['WinnerFormaLast10', 'LoserFormaLast10']:
    n_null = df[col].isna().sum()
    print(f"{col}: {n_null} nulls ({n_null/len(df):.2%}) -- rolling 10-match window needs >=1 prior match")

fig, ax = plt.subplots()
df['WinnerDaysSinceLastMatch'].clip(upper=180).hist(bins=60, ax=ax)
ax.set_title('Days since last match (winner side, clipped at 180)')
plt.tight_layout()
plt.show()


In [ ]:
for col in ['WinnerTrenutci', 'LoserTrenutci']:
    filled = df[col].notna().mean()
    print(f"{col}: {filled:.1%} filled")


## `Godiste` (birth year) quality

Recall from `../data/README.md`: `Igrac.DatumRodjenja` is 100% NULL in production — `Godiste`
(int birth year) is the real, usable age-proxy field. This section decides the plausibility
bounds used to clean it in notebook 02.


In [ ]:
for col in ['WinnerGodiste', 'LoserGodiste']:
    print(f"{col}: {df[col].notna().mean():.1%} filled")

godiste_long = pd.concat([df['WinnerGodiste'], df['LoserGodiste']]).dropna()
print()
print(godiste_long.describe())

GODISTE_LOWER, GODISTE_UPPER = 1930, 2015  # birth year -> age ~10-95 at match time
implausible = godiste_long[(godiste_long < GODISTE_LOWER) | (godiste_long > GODISTE_UPPER)]
print(f"\nImplausible Godiste values outside [{GODISTE_LOWER}, {GODISTE_UPPER}]: "
      f"{len(implausible)} of {len(godiste_long)} ({len(implausible)/len(godiste_long):.3%})")
print(sorted(implausible.unique()))

fig, ax = plt.subplots()
godiste_long.clip(lower=1900, upper=2100).hist(bins=80, ax=ax)
ax.axvline(GODISTE_LOWER, color='r', ls='--')
ax.axvline(GODISTE_UPPER, color='r', ls='--')
ax.set_title('Godiste distribution with chosen plausibility bounds (dashed red)')
plt.tight_layout()
plt.show()


**Decision carried into `02_feature_engineering.ipynb`:** derive age as
`YEAR(Datum) - Godiste`, using `Godiste` (not `DatumRodjenja`, which is unusable). Apply bounds
`[1930, 2015]` — this comfortably contains every real value seen (min real value 1947, max real
value 2007) while nulling out the handful of intentionally-wrong placeholder entries Marin
flagged (`0`, `42`, `81`, `7101976`, `21021985` — clearly typos/placeholders, not real birth
years).


## EDA Summary

- **17,128 rows, 47 columns**, `Datum` 2017-05-08 to present (`SusretId`/`Datum` ties broken by
  `SusretId` for a deterministic chronological order).
- **`LigaTip`:** SM (men) dominates (~91% of rows), SF (women) ~9% — expected given the club's
  membership composition.
- **Elo:** roughly bell-shaped, winner-side distribution visibly shifted right vs. loser-side (as
  expected — winners tend to have been the higher-rated player, though far from always).
- **Elo-favorite baseline:** the pre-match higher-rated player wins a majority but nowhere near
  all matches — the accuracy-vs-|Elo diff| bin table shows the classic "favorite is more likely to
  win, more so as the gap widens" shape, which is exactly the pattern Experiment 0 (the closed-form
  Elo formula) should already capture well, and the bar Experiments A/B need to clear.
- **H2H sparsity:** ~70% of matches have ZERO prior head-to-head meetings between the two players
  — H2H features MUST be paired with a shrinkage-toward-neutral treatment (done in notebook 02),
  otherwise the raw win-rate feature is mostly noise/overfitting bait on sparse pairs.
- **`LigaKoeficjent`:** 8 distinct levels (1=highest, 8=lowest — confirmed via monotonically
  decreasing average Elo per level).
- **`Lokacija`:** fully populated (100% non-null for in-scope league matches), only 17 distinct
  values, clean (no free-text variant problem) — the home-court feature should be usable with
  minimal normalization.
- **Season pyramid size (`BrojLigaUSezoni`/`BrojIgracaUkupno`):** `BrojLigaUSezoni` counts
  *groups*, not *levels* — a caveat that matters for the relative-pyramid-position feature (see
  callout above).
- **`DaysSinceLastMatch`/`FormaLast10` nulls:** both null on a player's first in-scope match (382
  winner-side / 397 loser-side rows) — expected, structural, not a data quality problem.
- **`Trenutci`:** ~82-88% filled — confirms Marin's expectation of high coverage (this is a v2,
  not-implemented-this-pass feature (that's Exp C4).
- **`Godiste`:** ~93% filled, 5 implausible placeholder values identified and bounded out via
  `[1930, 2015]` (decision documented above, applied in notebook 02).
